In [1]:
# FRACTATLAS DATASET SPLITTER
# Creates:
# train / val / test
# fractured / not_fractured

import os
import shutil
import random
from pathlib import Path
from sklearn.model_selection import train_test_split


# PATHS
BASE_DIR = Path(r"C:\Users\user\Downloads\AI Assessment\FracAtlas\FracAtlas")

FRACTURED_DIR = BASE_DIR / "images" / "Fractured"
NON_FRACTURED_DIR = BASE_DIR / "images" / "Non_fractured"

OUTPUT_DIR = Path(r"C:\Users\user\Downloads\AI Assessment\FracAtlas Dataset")

# SPLIT RATIOS
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_STATE = 42

# CREATE OUTPUT FOLDERS
splits = ["train", "val", "test"]
classes = ["fractured", "not_fractured"]

for split in splits:
    for cls in classes:
        os.makedirs(OUTPUT_DIR / split / cls, exist_ok=True)

print("Output folders created.")

# GET IMAGE FILES
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

fractured_images = [
    x for x in FRACTURED_DIR.iterdir()
    if x.suffix.lower() in IMAGE_EXTENSIONS
]

non_fractured_images = [
    x for x in NON_FRACTURED_DIR.iterdir()
    if x.suffix.lower() in IMAGE_EXTENSIONS
]

print(f"Fractured images: {len(fractured_images)}")
print(f"Non-fractured images: {len(non_fractured_images)}")

# SPLIT FUNCTION
def split_data(image_list):

    train_files, temp_files = train_test_split(
        image_list,
        test_size=(1 - TRAIN_RATIO),
        random_state=RANDOM_STATE,
        shuffle=True
    )

    val_files, test_files = train_test_split(
        temp_files,
        test_size=0.5,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    return train_files, val_files, test_files


# SPLIT FRACTURED
fractured_train, fractured_val, fractured_test = split_data(fractured_images)

# SPLIT NON-FRACTURED
non_train, non_val, non_test = split_data(non_fractured_images)

# COPY FILES
def copy_files(file_list, split_name, class_name):

    destination = OUTPUT_DIR / split_name / class_name

    for file_path in file_list:
        shutil.copy2(file_path, destination / file_path.name)

# Fractured
copy_files(fractured_train, "train", "fractured")
copy_files(fractured_val, "val", "fractured")
copy_files(fractured_test, "test", "fractured")

# Non-fractured
copy_files(non_train, "train", "not_fractured")
copy_files(non_val, "val", "not_fractured")
copy_files(non_test, "test", "not_fractured")

# PRINT FINAL COUNTS
print("\n==============================")
print("DATASET SPLIT COMPLETE")
print("==============================")

print("\nFractured:")
print(f"Train: {len(fractured_train)}")
print(f"Val  : {len(fractured_val)}")
print(f"Test : {len(fractured_test)}")

print("\nNon-Fractured:")
print(f"Train: {len(non_train)}")
print(f"Val  : {len(non_val)}")
print(f"Test : {len(non_test)}")

print("\nFinal Dataset Structure:")
print("""
FracAtlas Dataset/
 ├── train/
 │    ├── fractured/
 │    └── not_fractured/
 │
 ├── val/
 │    ├── fractured/
 │    └── not_fractured/
 │
 └── test/
      ├── fractured/
      └── not_fractured/
""")

Output folders created.
Fractured images: 717
Non-fractured images: 3366

DATASET SPLIT COMPLETE

Fractured:
Train: 501
Val  : 108
Test : 108

Non-Fractured:
Train: 2356
Val  : 505
Test : 505

Final Dataset Structure:

FracAtlas Dataset/
 ├── train/
 │    ├── fractured/
 │    └── not_fractured/
 │
 ├── val/
 │    ├── fractured/
 │    └── not_fractured/
 │
 └── test/
      ├── fractured/
      └── not_fractured/



In [2]:
import os
import random
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array, array_to_img

# PATH SETUP
FRACTURED_TRAIN_DIR = Path(r"C:\Users\user\Downloads\AI Assessment\FracAtlas Dataset\train\fractured")

TARGET_COUNT = 1200
IMG_SIZE = (224, 224)

# AUGMENTATION LAYERS
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.RandomContrast(0.10),
])

# GET ORIGINAL IMAGES ONLY
image_extensions = [".jpg", ".jpeg", ".png"]

original_images = [
    img for img in FRACTURED_TRAIN_DIR.iterdir()
    if img.suffix.lower() in image_extensions and not img.name.startswith("aug_")
]

current_count = len([
    img for img in FRACTURED_TRAIN_DIR.iterdir()
    if img.suffix.lower() in image_extensions
])

print(f"Original fractured images: {len(original_images)}")
print(f"Current fractured images : {current_count}")
print(f"Target fractured images  : {TARGET_COUNT}")

images_needed = TARGET_COUNT - current_count

if images_needed <= 0:
    print("No augmentation needed. Target already reached.")
else:
    print(f"Need to create {images_needed} augmented images.")

    created = 0

    while created < images_needed:
        img_path = random.choice(original_images)

        img = load_img(img_path, target_size=IMG_SIZE)
        img_array = img_to_array(img)
        img_array = tf.expand_dims(img_array, axis=0)

        augmented_img = data_augmentation(img_array, training=True)
        augmented_img = tf.squeeze(augmented_img, axis=0)

        augmented_img = tf.clip_by_value(augmented_img, 0, 255)
        augmented_img = array_to_img(augmented_img)

        save_name = f"aug_{created+1:04d}_{img_path.stem}.jpg"
        save_path = FRACTURED_TRAIN_DIR / save_name

        augmented_img.save(save_path)

        created += 1

    print(f"Created {created} augmented fractured images.")
    print(f"Final fractured train count should be around {TARGET_COUNT}.")

Original fractured images: 501
Current fractured images : 501
Target fractured images  : 1200
Need to create 699 augmented images.
Created 699 augmented fractured images.
Final fractured train count should be around 1200.


In [3]:
from pathlib import Path

fractured_count = len(list(Path(r"C:\Users\user\Downloads\AI Assessment\FracAtlas Dataset\train\fractured").glob("*")))
not_fractured_count = len(list(Path(r"C:\Users\user\Downloads\AI Assessment\FracAtlas Dataset\train\not_fractured").glob("*")))

print("Train fractured:", fractured_count)
print("Train not fractured:", not_fractured_count)

Train fractured: 1200
Train not fractured: 2356
